In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_recall_curve, roc_auc_score, classification_report
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv("creditcard.csv")

# Time-aware split (NO random shuffle!)
df['Time'] = df['Time'] / 3600
df = df.sort_values('Time').reset_index(drop=True)

# Feature groups
num_cols = ['Time', 'Amount']
pca_cols = [f'V{i}' for i in range(1, 29)]
target = 'Class'

print(f" Shape: {df.shape} | Fraud Rate: {df['Class'].mean():.4%}")

c:\Users\rajen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Shape: (284807, 31) | Fraud Rate: 0.1727%


In [3]:
# 1. Robust scaling for Amount (outlier-resistant)
scaler = RobustScaler()
df[['Amount_scaled']] = scaler.fit_transform(df[['Amount']])

# 2. Unsupervised Anomaly Scoring (Isolation Forest)
iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(df[pca_cols + ['Amount_scaled']]) # Fit the model
df['anomaly_score'] = iso.decision_function(df[pca_cols + ['Amount_scaled']])
df['is_anomaly'] = (df['anomaly_score'] < -0.1).astype(int)

# 3. Temporal Velocity Features (window-based risk signals)
df['hour'] = df['Time'].astype(int)
df['amount_log'] = np.log1p(df['Amount'])

# Rolling 1-hour transaction velocity (simulated via time bins)
hourly_agg = df.groupby('hour').agg(
    tx_count=('Class', 'count'),
    avg_amount=('amount_log', 'mean'),
    fraud_rate=('Class', 'mean')
).shift(1).fillna(method='bfill')

df = df.join(hourly_agg.add_prefix('rolling_1h_'), on='hour')

# Drop helper columns
df.drop(columns=['hour'], inplace=True)

print("✅ Engineered features: anomaly_score, is_anomaly, rolling_1h_*")

✅ Engineered features: anomaly_score, is_anomaly, rolling_1h_*


In [4]:
# Temporal split: Train on older data, validate on newer (prevents lookahead bias)
split_idx = int(len(df) * 0.8)
train, val = df.iloc[:split_idx], df.iloc[split_idx:]

X_train = train.drop(columns=['Class', 'Time', 'Amount'])
y_train = train['Class']
X_val = val.drop(columns=['Class', 'Time', 'Amount'])
y_val = val['Class']

# Drop rows with NaN values from validation sets to avoid XGBoost error
# Align indices after dropping NaNs to ensure X_val and y_val correspond
combined_val = pd.concat([X_val, y_val], axis=1)
combined_val.dropna(inplace=True)
X_val = combined_val.drop(columns=['Class'])
y_val = combined_val['Class']

# Cost-sensitive weight: False Negatives cost ~10x False Positives in real banking
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

# XGBoost with anomaly-augmented features
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
val_proba = xgb.predict_proba(X_val)[:, 1]

print(f"✅ Val ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")

✅ Val ROC-AUC: 0.9791


In [5]:
def calculate_financial_cost(y_true, y_pred, amounts, fp_cost=2.0, fn_recovery_rate=0.2):
    """
    Real-world cost function:
    - FP: Manual review cost (~$2/transaction)
    - FN: Unrecovered fraud loss (amount * (1 - recovery_rate))
    """
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn_amounts = amounts[(y_pred == 0) & (y_true == 1)]
    cost = (fp * fp_cost) + (fn_amounts * (1 - fn_recovery_rate)).sum()
    return cost, fp, len(fn_amounts)

# Find threshold that MINIMIZES financial loss
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)
# Align amounts_val with y_val by using the indices from the filtered y_val
amounts_val = val.loc[y_val.index, 'Amount'].values

min_cost = float('inf')
optimal_thresh = 0.5

for t in np.linspace(0.1, 0.9, 80):
    preds = (val_proba >= t).astype(int)
    cost, fp, fn = calculate_financial_cost(y_val.values, preds, amounts_val)
    if cost < min_cost:
        min_cost = cost
        optimal_thresh = t
        best_fp, best_fn = fp, fn

print(f" Optimal Threshold: {optimal_thresh:.3f}")
print(f"   Min Financial Loss: ${min_cost:,.2f}")
print(f"   False Positives: {best_fp} | False Negatives: {best_fn}")

# Apply optimal threshold
val_preds = (val_proba >= optimal_thresh).astype(int)
print(classification_report(y_val, val_preds, target_names=['Legit', 'Fraud']))

 Optimal Threshold: 0.505
   Min Financial Loss: $2,077.16
   False Positives: 81 | False Negatives: 16
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56887
       Fraud       0.42      0.79      0.55        75

    accuracy                           1.00     56962
   macro avg       0.71      0.89      0.77     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
# Global SHAP summary
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_val)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_val, feature_names=X_val.columns, show=False)
plt.title("Feature Impact on Fraud Prediction (SHAP)")
plt.tight_layout()
plt.show()

# Extract human-readable rules (Surrogate Decision Tree)
from sklearn.tree import DecisionTreeClassifier, export_text

surrogate = DecisionTreeClassifier(max_depth=3, random_state=42)
surrogate.fit(X_val, val_preds)

rules = export_text(surrogate, feature_names=list(X_val.columns))
print(" COMPLIANCE RULES (Surrogate Tree):")
print(rules)

In [ ]:
def simulate_roi(threshold, total_transactions=100000, fraud_rate=0.0017,
                 y_val_actual=None, val_proba_actual=None,
                 fp_cost=2.0, fn_recovery_rate=0.2):
    np.random.seed(42)

    if y_val_actual is not None and val_proba_actual is not None:
        # Apply the given threshold to the actual validation probabilities
        val_preds_for_sim = (val_proba_actual >= threshold).astype(int)

        # Calculate TP, FP, FN counts on the validation set for this threshold
        tp_val = ((val_preds_for_sim == 1) & (y_val_actual == 1)).sum()
        fp_val = ((val_preds_for_sim == 1) & (y_val_actual == 0)).sum()
        fn_val = ((val_preds_for_sim == 0) & (y_val_actual == 1)).sum()

        n_pos_val = y_val_actual.sum() # Actual number of frauds in validation set
        n_neg_val = len(y_val_actual) - n_pos_val # Actual number of legit in validation set

        # Calculate rates from validation set (handle division by zero)
        recall_rate_val = tp_val / n_pos_val if n_pos_val > 0 else 0
        fp_rate_val = fp_val / n_neg_val if n_neg_val > 0 else 0

        # Scale these rates to the total simulated transactions
        # total_transactions * fraud_rate is the simulated number of actual frauds
        # total_transactions * (1 - fraud_rate) is the simulated number of actual legit
        simulated_fn_count = (1 - recall_rate_val) * (total_transactions * fraud_rate)
        simulated_fp_count = fp_rate_val * (total_transactions * (1 - fraud_rate))

    else:
        # Fallback to original simplified simulation if no actual data provided
        print("Warning: Using simplified FP/FN estimation. Pass y_val_actual, val_proba_actual for refined estimation.")
        simulated_fn_count = int(total_transactions * fraud_rate * 0.15) # Fixed 85% caught
        simulated_fp_count = int(total_transactions * (threshold < 0.5) * 0.02 * total_transactions) # Simplified FP

    # Baseline loss calculation remains for comparison (original logic for baseline)
    baseline_fn_count = int(total_transactions * fraud_rate * 0.85)
    avg_fraud_amount = 125
    baseline_loss = baseline_fn_count * avg_fraud_amount

    # Model loss using dynamically calculated FP/FN (or simplified fallback)
    model_loss = (simulated_fn_count * avg_fraud_amount * (1 - fn_recovery_rate)) + \
                 (simulated_fp_count * fp_cost)
    savings = baseline_loss - model_loss

    # Ensure no division by zero for ROI calculation
    roi_denominator = (simulated_fp_count * fp_cost + 5000) # System cost added
    roi_str = f"{(savings/roi_denominator):.1f}x" if roi_denominator != 0 else "N/A"

    return {
        'Threshold': round(threshold, 3),
        'Fraud Caught': f"{recall_rate_val:.0%}" if 'recall_rate_val' in locals() else 'N/A',
        'False Positives': f"{int(simulated_fp_count):,}",
        'Monthly Savings': f"${int(savings):,}",
        'ROI vs Manual Review': roi_str
    }

roi_df = pd.DataFrame([
    simulate_roi(optimal_thresh, y_val_actual=y_val, val_proba_actual=val_proba),
    simulate_roi(0.5, y_val_actual=y_val, val_proba_actual=val_proba),
    simulate_roi(0.8, y_val_actual=y_val, val_proba_actual=val_proba)
])
print("\n BUSINESS IMPACT COMPARISON:")
print(roi_df.to_string(index=False))


📈 BUSINESS IMPACT COMPARISON:
 Threshold Fraud Caught False Positives Monthly Savings ROI vs Manual Review
     0.201         100%             252         $17,495                 3.2x
     0.500          95%             252         $16,686                 3.0x
     0.800          95%             224         $16,742                 3.1x
